In [1]:
from google.cloud import storage
import geopandas as gpd
from pyogrio.errors import DataSourceError
import pandas as pd
import os
import dotenv
dotenv.load_dotenv()
import matplotlib.pyplot as plt
import datetime as dt
import matplotlib.dates as mdates

In [2]:
OUTPUT_PATH = os.getenv("CSO_DATA_OUTPUT")
if OUTPUT_PATH is None:
    raise ValueError("Output Path not set")

In [ ]:
try:
    edm = pd.read_pickle(OUTPUT_PATH)
    print("Read edm from source")
except DataSourceError:
    edm = pd.DataFrame({
        # "geometry":[],
        "Monitor_ID": [],
        "Event_Start": [],
        "Event_End": [],
    })
    print("Output file doesn't exist, making a new Pandas DataFrame")

# NWG Data

In [5]:
INPUT_PATH = os.getenv("CSO_DATA_LOC")
if INPUT_PATH is None:
    raise ValueError("Input Path not set")

In [6]:
bucketName = INPUT_PATH.split("/")[2]
path_in_bucket = '/'.join(INPUT_PATH.split("/")[3:])

In [7]:
storage_client = storage.Client()
bucket = storage_client.get_bucket(bucketName)

In [40]:
geojson_filenames = ["gs://"+bucketName+'/' +
                     fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.geojson')]

In [8]:
csv_filenames = ["gs://"+bucketName+'/' +
                 fname.name for fname in bucket.list_blobs(prefix=path_in_bucket) if fname.name.endswith('.csv')]

In [ ]:
pd.read_csv(csv_filenames[0])

### Check if geometry remains same for all IDs

In [35]:
def add_set_to_setDf(id, colVal, setDf):
    if id not in setDf.index:
        setDf.loc[id, "check"] = {colVal}
    else:
        setDf.loc[id, "check"].add(colVal)
    return


def convert_to_set(x):
    if not isinstance(x, set):
        return {x}
    return x

In [38]:
def check_col(col: str, files: list[str] = geojson_filenames) -> pd.DataFrame:
    setDf = pd.DataFrame({"check": []
                          })
    setDf.index = setDf.index.astype('O')

    for filename in geojson_filenames:
        gdf = gpd.read_file(filename, columns=["id", col])
        gdf.apply(lambda x: add_set_to_setDf(x["id"], x[col], setDf), axis=1)
        setDf.check = setDf.check.swifter.apply(convert_to_set)
    return setDf

In [ ]:
setDf = check_col("geometry", files=geojson_filenames)

In [ ]:
setDf.check.apply(lambda x: len(x)).value_counts()

## Make EDM data

In [149]:
def add_to_set_EDM(start, end, id, setDf):
    if pd.isnull(start) or pd.isnull(end):
        return
    if id not in setDf.index:
        setDf.loc[id, "Event"] = {(start, end)}
    else:
        setDf.loc[id, "Event"].add((start, end))


def convert_to_EDM(startCol: str, endCol: str, files: list[str] = geojson_filenames) -> pd.DataFrame:
    setDf = pd.DataFrame({"Event": []})
    setDf.index = setDf.index.astype('O')

    for i, file in enumerate(files):
        df = pd.read_csv(file, usecols=["id", startCol, endCol], parse_dates=[
                         startCol, endCol])
        df.apply(lambda x: add_to_set_EDM(
            x[startCol], x[endCol], x["id"], setDf), axis=1)
        if i == 0:
            setDf.Event = setDf.Event.apply(convert_to_set)
    return setDf

In [157]:
startCol = "Latest Spill Start Time"
endCol = "Latest Spill End Time"

In [155]:
setDf = convert_to_EDM(startCol, endCol, csv_filenames[:])

In [160]:
exploded = setDf.explode("Event")
exploded["Event_Start"] = exploded["Event"].apply(lambda x: x[0]).dt.tz_localize("UTC")
exploded["Event_End"] = exploded["Event"].apply(lambda x: x[1]).dt.tz_localize("UTC")
exploded.index.rename("Monitor_ID", inplace=True)
exploded.drop(columns=["Event"], inplace=True)

In [ ]:
exploded.to_pickle(OUTPUT_PATH)

# Explore Data

In [ ]:
EDM_PATH = os.getenv("CSO_DATA_OUTPUT")
edm = pd.read_pickle("gs://rdmai_dev_data/cleansed/Theme2CD/CSO_EDM.pkl")

In [177]:
edm["Duration"] = edm["Event_End"] - edm["Event_Start"]

In [ ]:
nEvents = 50

fig, ax = plt.subplots()
ax.figure.set_size_inches(40, 20)
id_set = []

for index, row in edm.iloc[:nEvents].iterrows():
    if index not in id_set:
        id_set.append(index)
    y = 2*id_set.index(index)
    ax.add_patch(plt.Rectangle(
        (row['Event_Start'], y), row['Duration'], 1, fill=True, alpha=0.5,ec="black"))

for i in range(len(id_set)):
    ax.axhline(y=2*i+0.5, color='blue',ls="--")
    
ax.set_xlim(dt.datetime(2024, 11, 1), max(edm['Event_End']))
ax.set_ylim(-1, 2*len(id_set))

# format the x-axis to show dates
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))

ax.yaxis.set_major_locator(plt.MaxNLocator(len(id_set)))
# add a title and labels
ax.set_title('Event Timeline')
ax.set_xlabel('Date and Time')
ax.set_ylabel('Event ID')

# show the plot
plt.show()